# ERA5 Wind Hazard Data Pipeline — Central Asia (2015-2024)

This notebook downloads, processes and merges ERA5 reanalysis data for wind hazard assessment across five Central Asian countries.

---

## What is ERA5?

ERA5 is the fifth generation global atmospheric reanalysis produced by the **European Centre for Medium-Range Weather Forecasts (ECMWF)** under the **Copernicus Climate Change Service (C3S)**.

A reanalysis combines numerical weather prediction models with historical observations (weather stations, satellites, radiosondes) to produce a consistent and continuous reconstruction of past climate conditions.

| Property | Value |
|---|---|
| Spatial resolution | ~31 km (0.25 x 0.25 degrees) |
| Temporal resolution | Hourly |
| Temporal coverage | 1940 to present |
| Vertical levels | 137 atmospheric levels |
| Source | ECMWF / Copernicus C3S |

---

## Variables

### 10m_u_component_of_wind

- **Short name**: u10
- **Unit**: m/s
- **Step type**: Instant
- **Description**: Eastward component of wind at 10 metres above the surface. Positive values indicate wind blowing towards the East.
- **Relevance for wind hazard**: Combined with v10 to compute wind speed at 10m. Relevant for low-voltage lines and structures close to the ground.

### 10m_v_component_of_wind

- **Short name**: v10
- **Unit**: m/s
- **Step type**: Instant
- **Description**: Northward component of wind at 10 metres above the surface. Positive values indicate wind blowing towards the North.
- **Relevance for wind hazard**: Combined with u10 to compute wind speed at 10m: ws10 = sqrt(u10² + v10²)

### 100m_u_component_of_wind

- **Short name**: u100
- **Unit**: m/s
- **Step type**: Instant
- **Description**: Eastward component of wind at 100 metres above the surface.
- **Relevance for wind hazard**: More representative of wind loads experienced by high-voltage transmission towers, which can reach 50-100m in height. Wind speed is systematically higher at 100m than at 10m due to the vertical wind profile.

### 100m_v_component_of_wind

- **Short name**: v100
- **Unit**: m/s
- **Step type**: Instant
- **Description**: Northward component of wind at 100 metres above the surface.
- **Relevance for wind hazard**: Combined with u100 to compute wind speed at 100m: ws100 = sqrt(u100² + v100²)

---

## Wind Speed Calculation

ERA5 provides wind as two orthogonal components (u and v). Wind speed is derived as:

```
ws = sqrt(u² + v²)
```

This is computed in the indicator notebook after download. The raw u/v components are stored here.

---

## Time Sampling Strategy

This pipeline uses **4 time steps per day**: 00:00, 06:00, 12:00, 18:00 UTC.

For wind hazard on power line infrastructure:
- 4 steps/day captures the main diurnal wind cycle (night minimum, morning increase, afternoon peak, evening decrease)
- Sufficient to detect high-wind events and compute meaningful daily maxima
- Hourly data would be more precise but increases file size by 6x with marginal gain for regional-scale studies

---

## Study Area — Central Asia

| Country | Code | Bounding Box (N, W, S, E) |
|---|---|---|
| Tajikistan | TJK | 41.1 N, 67.3 E, 36.5 N, 75.2 E |
| Turkmenistan | TKM | 42.8 N, 52.2 E, 35.0 N, 66.8 E |
| Kyrgyzstan | KGZ | 43.3 N, 69.0 E, 39.0 N, 80.0 E |
| Kazakhstan | KAZ | 55.5 N, 46.0 E, 40.0 N, 87.5 E |
| Uzbekistan | UZB | 46.0 N, 55.0 E, 37.0 N, 74.0 E |

---

## Requirements

1. A registered account on the [Copernicus CDS](https://cds.climate.copernicus.eu/)
2. A configured `~/.cdsapirc` file containing your API credentials:
```
url: https://cds.climate.copernicus.eu/api
key: YOUR-API-KEY
```
3. Required packages: `cdsapi`, `xarray`, `h5netcdf`

---

## Pipeline Overview

1. Download ERA5 wind data per country per year via the CDS API
2. Unzip the returned archive and merge the variable files into one NetCDF per country per year
3. Merge all yearly files into a single file per country
4. Inspect and validate the output datasets

---
## 0. Imports and Parameters

In [1]:
import cdsapi
import os
import glob
import shutil
import zipfile
import numpy as np
import xarray as xr

DATA_DIR = "data/era5/wind"
YEARS    = [str(y) for y in range(2015, 2025)]
TIMES    = ["00:00", "06:00", "12:00", "18:00"]

VARIABLES = [
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "100m_u_component_of_wind",
    "100m_v_component_of_wind",
]

COUNTRIES = {
    "TJK": {"area": [41.1, 67.3, 36.5, 75.2]},
    "TKM": {"area": [42.8, 52.2, 35.0, 66.8]},
    "KGZ": {"area": [43.3, 69.0, 39.0, 80.0]},
    "KAZ": {"area": [55.5, 46.0, 40.0, 87.5]},
    "UZB": {"area": [46.0, 55.0, 37.0, 74.0]},
}

os.makedirs(DATA_DIR, exist_ok=True)

print("Parameters loaded")
print(f"  Countries  : {list(COUNTRIES.keys())}")
print(f"  Years      : {YEARS[0]} to {YEARS[-1]}")
print(f"  Time steps : {TIMES}")
print(f"  Variables  : {VARIABLES}")

Parameters loaded
  Countries  : ['TJK', 'TKM', 'KGZ', 'KAZ', 'UZB']
  Years      : 2015 to 2024
  Time steps : ['00:00', '06:00', '12:00', '18:00']
  Variables  : ['10m_u_component_of_wind', '10m_v_component_of_wind', '100m_u_component_of_wind', '100m_v_component_of_wind']


---
## 1. Download ERA5 Wind Data via CDS API

The CDS API may return either a ZIP archive or a direct NetCDF file depending on the request.
Since all 4 wind variables share the same step type (instant), the API often returns a single NetCDF directly.
The script detects the format automatically and handles both cases.

In [2]:
def is_zip(path):
    """Check if a file is a ZIP archive by reading its magic bytes."""
    with open(path, 'rb') as f:
        return f.read(4) == b'PK\x03\x04'


c = cdsapi.Client()

for country, params in COUNTRIES.items():
    for year in YEARS:
        final_file = os.path.join(DATA_DIR, f"era5_wind_{country}_{year}.nc")

        if os.path.exists(final_file):
            print(f"Skip: {country} {year} — file already exists")
            continue

        print(f"Downloading: {country} {year}")
        tmp_download = os.path.join(DATA_DIR, f"tmp_{country}_{year}.download")
        tmp_dir      = os.path.join(DATA_DIR, f"tmp_{country}_{year}")

        try:
            c.retrieve("reanalysis-era5-single-levels", {
                "product_type": "reanalysis",
                "variable": VARIABLES,
                "year": year,
                "month": [f"{m:02d}" for m in range(1, 13)],
                "day":   [f"{d:02d}" for d in range(1, 32)],
                "time":  TIMES,
                "area":  params["area"],
                "format": "netcdf",
            }, tmp_download)

            if is_zip(tmp_download):
                # API returned a ZIP — extract and merge NetCDF files inside
                print(f"  Format: ZIP")
                os.makedirs(tmp_dir, exist_ok=True)
                with zipfile.ZipFile(tmp_download, "r") as z:
                    z.extractall(tmp_dir)
                extracted = glob.glob(f"{tmp_dir}/*.nc")
                print(f"  Extracted: {[os.path.basename(f) for f in extracted]}")
                if len(extracted) == 1:
                    shutil.move(extracted[0], final_file)
                else:
                    datasets = [xr.open_dataset(f) for f in extracted]
                    xr.merge(datasets).to_netcdf(final_file)
                    for ds in datasets:
                        ds.close()
            else:
                # API returned a direct NetCDF — just rename
                print(f"  Format: NetCDF (direct)")
                shutil.move(tmp_download, final_file)

            print(f"  Saved: {final_file}")

        finally:
            if os.path.exists(tmp_download): os.remove(tmp_download)
            if os.path.exists(tmp_dir):      shutil.rmtree(tmp_dir)

print("Download complete")

Skip: TJK 2015 — file already exists
Skip: TJK 2016 — file already exists
Skip: TJK 2017 — file already exists
Skip: TJK 2018 — file already exists
Skip: TJK 2019 — file already exists
Skip: TJK 2020 — file already exists
Skip: TJK 2021 — file already exists
Skip: TJK 2022 — file already exists
Skip: TJK 2023 — file already exists
Skip: TJK 2024 — file already exists
Skip: TKM 2015 — file already exists
Skip: TKM 2016 — file already exists
Skip: TKM 2017 — file already exists
Skip: TKM 2018 — file already exists
Skip: TKM 2019 — file already exists
Skip: TKM 2020 — file already exists
Skip: TKM 2021 — file already exists
Skip: TKM 2022 — file already exists
Skip: TKM 2023 — file already exists
Skip: TKM 2024 — file already exists
Skip: KGZ 2015 — file already exists
Skip: KGZ 2016 — file already exists
Skip: KGZ 2017 — file already exists
Skip: KGZ 2018 — file already exists
Skip: KGZ 2019 — file already exists
Skip: KGZ 2020 — file already exists
Skip: KGZ 2021 — file already exists
S

---
## 2. Verify Downloaded Files

In [3]:
files = sorted(glob.glob(f"{DATA_DIR}/era5_wind_*_20*.nc"))
print(f"{len(files)} files found:\n")
for f in files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {os.path.basename(f):<40} {size_mb:.1f} MB")

50 files found:

  era5_wind_KAZ_2015.nc                    144.3 MB
  era5_wind_KAZ_2016.nc                    145.5 MB
  era5_wind_KAZ_2017.nc                    144.8 MB
  era5_wind_KAZ_2018.nc                    146.0 MB
  era5_wind_KAZ_2019.nc                    144.8 MB
  era5_wind_KAZ_2020.nc                    145.3 MB
  era5_wind_KAZ_2021.nc                    145.0 MB
  era5_wind_KAZ_2022.nc                    144.7 MB
  era5_wind_KAZ_2023.nc                    144.8 MB
  era5_wind_KAZ_2024.nc                    145.3 MB
  era5_wind_KGZ_2015.nc                    11.0 MB
  era5_wind_KGZ_2016.nc                    11.1 MB
  era5_wind_KGZ_2017.nc                    11.1 MB
  era5_wind_KGZ_2018.nc                    11.1 MB
  era5_wind_KGZ_2019.nc                    11.0 MB
  era5_wind_KGZ_2020.nc                    11.1 MB
  era5_wind_KGZ_2021.nc                    11.1 MB
  era5_wind_KGZ_2022.nc                    11.0 MB
  era5_wind_KGZ_2023.nc                    11.1 MB
  er

---
## 3. Merge Yearly Files by Country

Concatenate all 10 yearly files (2015-2024) for each country into a single NetCDF file.

In [4]:
for country in COUNTRIES:
    output_file = f"{DATA_DIR}/era5_wind_{country}.nc"

    if os.path.exists(output_file):
        print(f"Skip: {country} — merged file already exists")
        continue

    print(f"Merging: {country}")
    files = sorted(glob.glob(f"{DATA_DIR}/era5_wind_{country}_20*.nc"))

    if not files:
        print(f"  Warning: no files found for {country}")
        continue

    merged = xr.open_mfdataset(files, combine="by_coords")
    merged.to_netcdf(output_file)
    print(f"  Saved: {output_file}")

print("Merge complete")

Skip: TJK — merged file already exists
Skip: TKM — merged file already exists
Skip: KGZ — merged file already exists
Skip: KAZ — merged file already exists
Skip: UZB — merged file already exists
Merge complete


---
## 4. Inspect Merged Datasets

Also computes wind speed from u/v components as a quick validation.
Expected pattern: Kazakhstan and Kyrgyzstan should show highest wind speeds due to steppe and mountain terrain.

In [5]:
for country in COUNTRIES:
    path = f"{DATA_DIR}/era5_wind_{country}.nc"
    if not os.path.exists(path):
        continue

    ds = xr.open_dataset(path)
    time_dim = "valid_time" if "valid_time" in ds.sizes else "time"

    print(f"\n{country}")
    print(f"  Variables      : {list(ds.data_vars)}")
    print(f"  Dimensions     : {dict(ds.sizes)}")
    print(f"  Period         : {str(ds[time_dim].values[0])[:10]} to {str(ds[time_dim].values[-1])[:10]}")
    print(f"  Time steps     : {len(ds[time_dim])}")

    # Wind speed at 10m
    if "u10" in ds and "v10" in ds:
        ws10 = np.sqrt(ds["u10"]**2 + ds["v10"]**2)
        print(f"  WS10 mean      : {float(ws10.mean()):.2f} m/s")
        print(f"  WS10 max       : {float(ws10.max()):.2f} m/s")
        print(f"  WS10 > 15 m/s  : {int((ws10 > 15).sum())} timesteps ({100*float((ws10 > 15).mean()):.1f}%)")
        print(f"  WS10 > 25 m/s  : {int((ws10 > 25).sum())} timesteps")

    # Wind speed at 100m
    if "u100" in ds and "v100" in ds:
        ws100 = np.sqrt(ds["u100"]**2 + ds["v100"]**2)
        print(f"  WS100 mean     : {float(ws100.mean()):.2f} m/s")
        print(f"  WS100 max      : {float(ws100.max()):.2f} m/s")
        print(f"  WS100 > 20 m/s : {int((ws100 > 20).sum())} timesteps ({100*float((ws100 > 20).mean()):.1f}%)")

    ds.close()

/opt/anaconda3/envs/ri-infra/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)



TJK
  Variables      : ['u10', 'v10', 'u100', 'v100']
  Dimensions     : {'valid_time': 14612, 'latitude': 19, 'longitude': 31}
  Period         : 2015-01-01 to 2024-12-31
  Time steps     : 14612
  WS10 mean      : 1.43 m/s
  WS10 max       : 15.56 m/s
  WS10 > 15 m/s  : 2 timesteps (0.0%)
  WS10 > 25 m/s  : 0 timesteps
  WS100 mean     : 2.25 m/s
  WS100 max      : 21.44 m/s
  WS100 > 20 m/s : 10 timesteps (0.0%)

TKM
  Variables      : ['u10', 'v10', 'u100', 'v100']
  Dimensions     : {'valid_time': 14612, 'latitude': 32, 'longitude': 59}
  Period         : 2015-01-01 to 2024-12-31
  Time steps     : 14612
  WS10 mean      : 3.72 m/s
  WS10 max       : 20.64 m/s
  WS10 > 15 m/s  : 2721 timesteps (0.0%)
  WS10 > 25 m/s  : 0 timesteps
  WS100 mean     : 5.34 m/s
  WS100 max      : 26.09 m/s
  WS100 > 20 m/s : 1294 timesteps (0.0%)

KGZ
  Variables      : ['u10', 'v10', 'u100', 'v100']
  Dimensions     : {'valid_time': 14612, 'latitude': 18, 'longitude': 45}
  Period         : 2015-01

---
## 5. Output Summary — Downloaded Data

In [6]:
print("Merged files by country:\n")
for country in COUNTRIES:
    path = f"{DATA_DIR}/era5_wind_{country}.nc"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  OK      era5_wind_{country}.nc — {size_mb:.1f} MB")
    else:
        print(f"  MISSING era5_wind_{country}.nc")

Merged files by country:

  OK      era5_wind_TJK.nc — 83.5 MB
  OK      era5_wind_TKM.nc — 260.1 MB
  OK      era5_wind_KGZ.nc — 113.4 MB
  OK      era5_wind_KAZ.nc — 1441.2 MB
  OK      era5_wind_UZB.nc — 418.0 MB


---
# Part 2 — Wind Hazard Indicators

This section computes wind hazard indicators from the merged ERA5 datasets.
For each country and each year, four indicators are derived from the u/v wind components at 10m and 100m.

## Indicators

| Indicator | Code | Definition | Height | Relevance |
|---|---|---|---|---|
| Max Wind Speed | MWS | Maximum wind speed (m/s) over the year | 100m | Peak mechanical load on towers and conductors |
| High Wind Days | HWD | Days/year with at least one timestep > 15 m/s | 10m | Frequency of operationally significant wind events |
| Storm Days | STD | Days/year with at least one timestep > 25 m/s | 100m | Extreme events likely to cause structural damage |
| Mean Wind Speed | MNW | Mean annual wind speed (m/s) | 100m | Baseline wind climate, useful for fatigue assessment |

All indicators are computed per year, then exported as a single NetCDF per country with a `year` dimension.
The output file per country is: `data/era5/wind/indicators/era5_wind_indicators_{COUNTRY}.nc`

---
## 6. Indicator Parameters

In [7]:
IND_DIR = "data/era5/wind/indicators"
os.makedirs(IND_DIR, exist_ok=True)

# Thresholds
THRESH_HWD = 15.0   # m/s at 10m — High Wind Days
THRESH_STD = 25.0   # m/s at 100m — Storm Days

print(f"Indicators output dir : {os.path.abspath(IND_DIR)}")
print(f"High Wind Days threshold  : ws10 > {THRESH_HWD} m/s")
print(f"Storm Days threshold      : ws100 > {THRESH_STD} m/s")

Indicators output dir : /Users/nassimdekkar/Documents/RI-Infra-exposure/notebooks/data/era5/wind/indicators
High Wind Days threshold  : ws10 > 15.0 m/s
Storm Days threshold      : ws100 > 25.0 m/s


---
## 7. Compute Indicators per Country

In [ ]:
def get_time_dim(ds):
    return "valid_time" if "valid_time" in ds.sizes else "time"


def compute_indicators_for_year(ds_year, time_dim):
    """Compute the 4 wind indicators for a single year dataset."""

    # Wind speed at 10m and 100m
    ws10  = np.sqrt(ds_year["u10"] **2 + ds_year["v10"] **2)
    ws100 = np.sqrt(ds_year["u100"]**2 + ds_year["v100"]**2)

    # --- MWS: max wind speed at 100m over the year ---
    mws = ws100.max(dim=time_dim).astype("float32")
    mws.name = "max_wind_speed"
    mws.attrs = {"long_name": "Max Wind Speed (100m)", "units": "m/s"}

    # --- MNW: mean wind speed at 100m over the year ---
    mnw = ws100.mean(dim=time_dim).astype("float32")
    mnw.name = "mean_wind_speed"
    mnw.attrs = {"long_name": "Mean Wind Speed (100m)", "units": "m/s"}

    # --- HWD: days with ws10 > threshold ---
    # Group timesteps by date, flag day if any timestep exceeds threshold
    ws10_daily_max = ws10.resample({time_dim: "1D"}).max()
    hwd = (ws10_daily_max > THRESH_HWD).sum(dim=time_dim).astype("float32")
    hwd.name = "high_wind_days"
    hwd.attrs = {"long_name": f"High Wind Days (ws10 > {THRESH_HWD} m/s)", "units": "days/year"}

    # --- STD: days with ws100 > storm threshold ---
    ws100_daily_max = ws100.resample({time_dim: "1D"}).max()
    std = (ws100_daily_max > THRESH_STD).sum(dim=time_dim).astype("float32")
    std.name = "storm_days"
    std.attrs = {"long_name": f"Storm Days (ws100 > {THRESH_STD} m/s)", "units": "days/year"}

    return mws, mnw, hwd, std


for country in COUNTRIES:
    nc_path = f"{DATA_DIR}/era5_wind_{country}.nc"
    out_path = f"{IND_DIR}/era5_wind_indicators_{country}.nc"

    if not os.path.exists(nc_path):
        print(f"MISSING: {nc_path} — skip")
        continue

    if os.path.exists(out_path):
        print(f"Skip: {country} — indicators already exist")
        continue

    print(f"\nComputing indicators: {country}")
    ds = xr.open_dataset(nc_path)
    time_dim = get_time_dim(ds)

    mws_list, mnw_list, hwd_list, std_list = [], [], [], []
    year_coords = []

    for year, ds_year in ds.groupby(f"{time_dim}.year"):
        print(f"  {year}", end=" ", flush=True)
        mws, mnw, hwd, std = compute_indicators_for_year(ds_year, time_dim)
        mws_list.append(mws)
        mnw_list.append(mnw)
        hwd_list.append(hwd)
        std_list.append(std)
        year_coords.append(year)

    print()

    def stack(arrays):
        return xr.concat(arrays, dim=xr.DataArray(year_coords, dims="year", name="year"))

    ds_out = xr.Dataset({
        "max_wind_speed":  stack(mws_list),
        "mean_wind_speed": stack(mnw_list),
        "high_wind_days":  stack(hwd_list),
        "storm_days":      stack(std_list),
    })
    ds_out.to_netcdf(out_path)
    ds.close()
    print(f"  Saved: {out_path}")

print("\nIndicator computation complete")


Computing indicators: TJK
  2015   2016   2017   2018   2019   2020   2021   2022   2023   2024 
  Saved: data/era5/wind/indicators/era5_wind_indicators_TJK.nc

Computing indicators: TKM
  2015   2016   2017   2018   2019   2020   2021   2022   2023   2024 
  Saved: data/era5/wind/indicators/era5_wind_indicators_TKM.nc

Computing indicators: KGZ
  2015   2016   2017   2018   2019   2020   2021   2022   2023   2024 
  Saved: data/era5/wind/indicators/era5_wind_indicators_KGZ.nc

Computing indicators: KAZ
  2015   2016 

---
## 8. Validate Indicators

In [ ]:
print(f"{'Country':<6} {'Indicator':<20} {'Mean':>8} {'Min':>8} {'Max':>8} {'Units'}")
print("-" * 65)

for country in COUNTRIES:
    path = f"{IND_DIR}/era5_wind_indicators_{country}.nc"
    if not os.path.exists(path):
        continue
    ds = xr.open_dataset(path)
    for var in ds.data_vars:
        da = ds[var].mean(dim="year").values
        units = ds[var].attrs.get("units", "")
        print(f"{country:<6} {var:<20} {float(np.nanmean(da)):>8.2f} {float(np.nanmin(da)):>8.2f} {float(np.nanmax(da)):>8.2f}  {units}")
    ds.close()
    print()

---
## 9. Output Summary

In [ ]:
print("Downloaded data:\n")
for country in COUNTRIES:
    path = f"{DATA_DIR}/era5_wind_{country}.nc"
    status = f"OK  — {os.path.getsize(path)/1024/1024:.1f} MB" if os.path.exists(path) else "MISSING"
    print(f"  era5_wind_{country}.nc            {status}")

print("\nIndicators:\n")
for country in COUNTRIES:
    path = f"{IND_DIR}/era5_wind_indicators_{country}.nc"
    status = f"OK  — {os.path.getsize(path)/1024/1024:.1f} MB" if os.path.exists(path) else "MISSING"
    print(f"  era5_wind_indicators_{country}.nc  {status}")

print("\nNext step: integrate wind hazard into the multi-hazard pipeline (wind_module.py)")